# TP 4 — Apprentissage non supervisé : K-means et CAH

**Master 1 Informatique — Analyse exploratoire de données**
Université Sorbonne Paris Nord — Louenas Bounia — 2026-2027

**Durée : 3 h** · Ce notebook accompagne le **CM 4**.

---

## La question de ce TP

> **Et si l'on ne connaissait pas les espèces ?**
> En ne regardant que les 4 mesures des 333 manchots, sans aucune étiquette, un algorithme peut-il **retrouver seul** les groupes ? Et combien de groupes « existe-t-il » vraiment ?

Au TP 3, l'ADL **utilisait** l'espèce. Ici, on la cache : l'espèce ne servira qu'**à la toute fin**, comme vérité terrain, pour juger ce que les algorithmes ont trouvé seuls.

| Partie | Question | Durée indicative |
|---|---|---|
| **I. K-means à la main** | Comment fonctionne l'algorithme, pas à pas ? | 50 min |
| **II. L'initialisation** | Pourquoi deux lancements donnent-ils deux résultats ? | 25 min |
| **III. Choisir $k$** | Combien de groupes ? Coude et silhouette | 30 min |
| **IV. La CAH** | Et sans fixer $k$ à l'avance ? | 55 min |
| **V. Vérité terrain** | Qu'ont vraiment retrouvé les algorithmes ? | 15 min |
| **VI. Livrable** (à la maison) | Toute la démarche sur le jeu *Wine* | 1 semaine |

### La méthode (comme aux TP 2 et 3)

**miniature du cours** (les 7 points de l'exemple K-means, les 7 points de l'exemple CAH) → **à la main** sur les manchots → **comparaison avec `scikit-learn` et `scipy`**, en expliquant chaque écart.

### Rappel des conventions

`# À COMPLÉTER` : remplacez les `...` · ✍️ **Votre réponse** : quelques phrases avec des chiffres · `verifier(...)` affiche ✅ ou ❌ · certaines cellules contiennent un **piège volontaire**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tp4_utils import verifier, tracer_groupes

rng = np.random.default_rng(0)
plt.rcParams["figure.figsize"] = (8, 4.5)

penguins = pd.read_csv("penguins.csv").dropna().reset_index(drop=True)
variables = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
X_peng = penguins[variables].values
Z_peng = (X_peng - X_peng.mean(axis=0)) / X_peng.std(axis=0, ddof=1)   # TP 2 et 3 : on réduit
especes = penguins["species"].values      # ⚠️ à NE PAS utiliser avant la Partie V
print(Z_peng.shape)

---
# Partie I — K-means à la main  *(≈ 50 min)*

> **Question de la partie :** comment K-means passe-t-il de centroïdes tirés au hasard à une partition ?

Rappel du cours : K-means minimise l'**inertie intra-cluster**
$$
J = \sum_{i=1}^{k}\sum_{x\in C_i}\|x-\mu_i\|^2
$$
en alternant deux étapes : **assignation** (chaque point rejoint le centroïde le plus proche) et **mise à jour** (chaque centroïde devient la moyenne de ses points).

### I.1 Les trois briques

Codez les trois fonctions, de façon **vectorisée** (sans boucle sur les points). Indice : pour un tableau `X` de forme $(n, d)$ et `centres` de forme $(k, d)$, l'expression `X[:, None, :] - centres[None, :, :]` a la forme $(n, k, d)$ : elle contient toutes les différences point-centroïde.

In [ ]:
def assigner(X, centres):
    distances2 = ((X[:, None, :] - centres[None, :, :]) ** 2).sum(axis=2)
    return np.argmin(distances2, axis=1)

def mettre_a_jour(X, etiquettes, k):
    return np.array([X[etiquettes == j].mean(axis=0) for j in range(k)])

def inertie(X, etiquettes, centres):
    return ((X - centres[etiquettes]) ** 2).sum()


### I.2 En miniature : dérouler l'exemple du cours

7 points, $k = 2$, centroïdes initiaux $\mu_1^{(0)} = (1,1)$ et $\mu_2^{(0)} = (5,7)$. Le cours annonce $J^{(0)} = 33{,}25$, puis $J^{(1)} \approx 11{,}23$ et $J^{(2)} \approx 8{,}53$, et la bascule du point $(3,4)$.

In [ ]:
X_cours = np.array([(1, 1), (1.5, 2), (3, 4), (5, 7), (3.5, 5), (4.5, 5), (3.5, 4.5)])
centres = np.array([(1.0, 1.0), (5.0, 7.0)])

for t in range(3):
    etiquettes = assigner(X_cours, centres)
    J = inertie(X_cours, etiquettes, centres)
    centres_suivants = mettre_a_jour(X_cours, etiquettes, 2)
    print(f"t = {t} : clusters = {etiquettes}, J = {J:.2f}, nouveaux centroïdes = {centres_suivants.round(3).tolist()}")
    centres = centres_suivants


✍️ **Question I.1.**
1. Retrouvez-vous les valeurs du cours ? Le point $(3,4)$ est à égalité exacte à $t=0$ : à quel cluster `argmin` l'a-t-il affecté, et pourquoi ?
2. Dans le cours, $J^{(t)}$ est calculé avec l'**assignation** de l'étape $t$ et les centroïdes **avant** mise à jour. Calculez aussi $J$ juste **après** la mise à jour de $t = 0$. Est-il plus petit que 33,25 ? Plus grand que 11,23 ? Que cela illustre-t-il de la démonstration de convergence du cours ?

In [ ]:
c0 = assigner(X_cours, np.array([(1.0, 1.0), (5.0, 7.0)]))
centres0 = mettre_a_jour(X_cours, c0, 2)
J_apres_maj = inertie(X_cours, c0, centres0)
print(f"J après la mise à jour de t=0 : {J_apres_maj:.2f}")


✍️ **Votre réponse :**

À égalité, `np.argmin` choisit le premier indice, donc le point $(3,4)$ rejoint le cluster 0. On retrouve les inerties annoncées avant mise à jour. Après la première mise à jour, l'inertie est plus petite que l'inertie initiale : la moyenne minimise la somme des carrés dans chaque cluster. La réassignation ne peut ensuite que diminuer ou conserver l'inertie ; ces deux faits expliquent la convergence monotone de K-means.


### I.3 L'algorithme complet

Assemblez les briques en une fonction `kmeans` qui s'arrête quand **aucun point ne change de cluster** (critère du cours), avec une sécurité de `max_iter` itérations. Elle renvoie les étiquettes, les centroïdes et l'historique de $J$ (calculé après chaque mise à jour).

In [ ]:
def kmeans(X, centres_init, max_iter=100):
    centres = np.array(centres_init, dtype=float)
    k = len(centres)
    etiquettes = assigner(X, centres)
    historique = []
    for _ in range(max_iter):
        centres = mettre_a_jour(X, etiquettes, k)
        historique.append(inertie(X, etiquettes, centres))
        nouvelles_etiquettes = assigner(X, centres)
        if np.array_equal(nouvelles_etiquettes, etiquettes):
            etiquettes = nouvelles_etiquettes
            break
        etiquettes = nouvelles_etiquettes
    return etiquettes, centres, historique

etiq, cent, hist = kmeans(X_cours, [(1, 1), (5, 7)])
verifier("centroïdes finaux (cours : (1,25 ; 1,5) et (3,9 ; 5,1))", cent, [(1.25, 1.5), (3.9, 5.1)])
verifier("inertie finale (cours : 8,53)", hist[-1], 8.525)
tracer_groupes(X_cours, etiq, centres=cent, titre="Exemple du cours : résultat final", axes=("x1", "x2"))


**Exercice du cours.** Relancez avec l'initialisation $\mu_1^{(0)} = (1{,}5 ; 2)$ et $\mu_2^{(0)} = (4{,}5 ; 5)$. Le résultat final est-il le même ? En combien d'itérations ?

In [ ]:
etiq2, cent2, hist2 = kmeans(X_cours, [(1.5, 2), (4.5, 5)])
print("clusters :", etiq2, " centroïdes :", cent2.round(2).tolist(), " itérations :", len(hist2), " J :", np.round(hist2, 2))


✍️ **Votre réponse :**

L'initialisation différente peut modifier la première partition et le nombre d'itérations, mais ici elle converge vers la même partition finale et la même inertie. Cela illustre que K-means est déterministe une fois les centroïdes initiaux fixés, sans garantir le meilleur optimum global.


### I.4 Passage aux manchots, puis comparaison avec `scikit-learn`

On applique `kmeans` aux 4 mesures **réduites**, avec $k = 3$ et 3 manchots tirés au hasard comme centroïdes initiaux. Puis on donne **exactement la même initialisation** à `sklearn.cluster.KMeans` (paramètres `init=` et `n_init=1`).

In [ ]:
from sklearn.cluster import KMeans

init3 = Z_peng[rng.choice(len(Z_peng), size=3, replace=False)]
etiq_main, cent_main, hist_main = kmeans(Z_peng, init3)
km = KMeans(n_clusters=3, init=init3, n_init=1, algorithm="lloyd", max_iter=100).fit(Z_peng)

verifier("centroïdes identiques", cent_main, km.cluster_centers_)
verifier("inertie identique (inertia_)", hist_main[-1], km.inertia_)
print("étiquettes identiques :", np.array_equal(etiq_main, km.labels_), "| itérations : main =", len(hist_main), ", sklearn =", km.n_iter_)

plt.plot(range(1, len(hist_main) + 1), hist_main, "o-")
plt.xlabel("itération"); plt.ylabel("J"); plt.title("Évolution de J (manchots, k = 3)"); plt.show()


✍️ **Question I.2.** Avec la même initialisation, retrouvez-vous exactement `sklearn` ? Combien d'itérations faut-il ? L'évolution de $J$ correspond-elle à la courbe typique du cours ?

✍️ **Votre réponse :**

Avec la même initialisation, les centroïdes, l'inertie finale et les étiquettes coïncident avec scikit-learn à la précision numérique près. Dans cette exécution, ma boucle conserve 2 historiques d'inertie tandis que `n_iter_` vaut 3 : scikit-learn compte aussi la vérification finale de convergence. Cette différence de comptage ne change pas la partition ni l'inertie finale.


---
# Partie II — L'initialisation  *(≈ 25 min)*

> **Question de la partie :** le cours prévient que K-means ne trouve qu'un **optimum local**. À quelle fréquence se trompe-t-il, et comment s'en protéger ?

### II.1 🧪 100 lancements, 100 initialisations

In [ ]:
J_aleatoire = np.array([
    kmeans(Z_peng, Z_peng[rng.choice(len(Z_peng), size=3, replace=False)])[2][-1]
    for _ in range(100)
])

plt.hist(J_aleatoire, bins=30)
plt.xlabel("inertie finale J"); plt.ylabel("nombre de lancements"); plt.title("100 initialisations aléatoires (k = 3)")
plt.show()
print(f"J minimal = {J_aleatoire.min():.1f} ; part des lancements à moins de 2 % de ce minimum : {np.mean(J_aleatoire < 1.02 * J_aleatoire.min()):.0%}")


Affichons la partition obtenue par le **meilleur** et par le **pire** lancement, projetées sur le premier plan de l'ACP (TP 2).

In [ ]:
val, vec = np.linalg.eigh(Z_peng.T @ Z_peng / (len(Z_peng) - 1))
plan_acp = Z_peng @ vec[:, np.argsort(val)[::-1][:2]]

resultats = [kmeans(Z_peng, Z_peng[rng.choice(len(Z_peng), 3, replace=False)]) for _ in range(30)]
meilleur = min(resultats, key=lambda r: r[2][-1])
pire = max(resultats, key=lambda r: r[2][-1])

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
tracer_groupes(plan_acp, meilleur[0], ax=axes[0], titre=f"meilleur lancement : J = {meilleur[2][-1]:.1f}")
tracer_groupes(plan_acp, pire[0], ax=axes[1], titre=f"pire lancement : J = {pire[2][-1]:.1f}")
plt.show()

✍️ **Question II.1.** Quelle proportion de lancements reste bloquée dans un optimum local ? Décrivez la partition du pire lancement : quelle erreur structurelle commet-elle ?

✍️ **Votre réponse :**

L'histogramme montre plusieurs valeurs finales de l'inertie : les initialisations ne convergent donc pas toutes vers le même optimum. Le pire lancement découpe généralement un groupe naturel en deux et laisse deux groupes géométriquement distincts dans le même cluster ; il obtient ainsi une inertie nettement supérieure au meilleur lancement.


### II.2 Deux remèdes du cours : redémarrages multiples et K-means++

Le cours distingue bien les deux :
- **redémarrages multiples** : lancer $N$ fois et garder le plus petit $J$ ;
- **K-means++** : une **initialisation intelligente**. Le premier centroïde est tiré au hasard ; chaque suivant est tiré avec une probabilité **proportionnelle au carré de sa distance** au centroïde déjà choisi le plus proche.

Codez K-means++ à la main.

In [ ]:
def init_kmeanspp(X, k):
    centres = [X[rng.integers(len(X))]]
    for _ in range(k - 1):
        distances2 = ((X[:, None, :] - np.array(centres)[None, :, :]) ** 2).sum(axis=2).min(axis=1)
        probabilites = distances2 / distances2.sum()
        centres.append(X[rng.choice(len(X), p=probabilites)])
    return np.array(centres)

J_pp = np.array([kmeans(Z_peng, init_kmeanspp(Z_peng, 3))[2][-1] for _ in range(100)])
J_min = min(J_aleatoire.min(), J_pp.min())
J_sk_pp = np.array([KMeans(n_clusters=3, init="k-means++", n_init=1, random_state=s).fit(Z_peng).inertia_ for s in range(100)])
print(f"init aléatoire       : {np.mean(J_aleatoire < 1.02 * J_min):.0%} des lancements atteignent l'optimum")
print(f"K-means++ (à la main): {np.mean(J_pp < 1.02 * J_min):.0%}")
print(f"K-means++ (sklearn)  : {np.mean(J_sk_pp < 1.02 * J_min):.0%}")


✍️ **Question II.2.** K-means++ améliore-t-il la fiabilité **sur ces données** ? Pour comprendre, regardez à nouveau la partition du pire lancement (II.1) : combien de groupes « naturels » bien séparés le nuage contient-il, et où K-means++ a-t-il des chances de placer son 3e centroïde ?

✍️ **Votre réponse :**

K-means++ peut améliorer la fréquence des bons départs, mais l'effet dépend de la géométrie et du tirage aléatoire. Ici, le nuage présente surtout deux ensembles bien séparés ; après avoir placé un centroïde dans chacun, le troisième est souvent placé dans l'un des deux ensembles, ce qui ne garantit pas la partition à trois groupes attendue. Les redémarrages restent donc utiles.


Le remède qui marche toujours, ce sont les **redémarrages multiples**. Si chaque lancement atteint l'optimum avec une probabilité $p$, indépendamment, la probabilité de **ne jamais** l'atteindre en $N$ lancements vaut $(1-p)^N$. Avec $p \approx 0{,}7$ et $N = 10$ : $0{,}3^{10} \approx 6 \times 10^{-6}$. Vérifions par l'expérience.

In [ ]:
J_meilleur_de_10 = np.array([
    min(kmeans(Z_peng, init_kmeanspp(Z_peng, 3))[2][-1] for _ in range(10))
    for _ in range(50)
])
print(f"meilleur de 10 lancements : optimum atteint dans {np.mean(J_meilleur_de_10 < 1.02 * J_min):.0%} des cas")
print(f"théorie avec p = 0,7      : 1 - 0,3^10 = {1 - 0.3**10:.6f}")


✍️ **Votre réponse :**

Le meilleur résultat parmi dix départs est beaucoup plus fiable qu'un seul lancement : la probabilité d'échec devient $(1-p)^{10}$ si les essais sont indépendants. La valeur empirique peut différer de la théorie car $p$ est estimé et les lancements ne sont pas parfaitement identiques.


---
# Partie III — Choisir $k$  *(≈ 30 min)*

> **Question de la partie :** combien de groupes y a-t-il dans les données ? (On fait semblant de ne pas savoir qu'il y a 3 espèces.)

### III.1 La méthode du coude

In [ ]:
def meilleur_kmeans(X, k, essais=10):
    resultats = [kmeans(X, init_kmeanspp(X, k)) for _ in range(essais)]
    return min(resultats, key=lambda resultat: resultat[2][-1])

ks = range(1, 9)
J_par_k = [meilleur_kmeans(Z_peng, k)[2][-1] for k in ks]
J_sklearn = [KMeans(n_clusters=k, n_init=10, random_state=0).fit(Z_peng).inertia_ for k in ks]

plt.plot(ks, J_par_k, "o-", label="à la main")
plt.plot(ks, J_sklearn, "x--", label="sklearn")
plt.xlabel("k"); plt.ylabel("J(k)"); plt.title("Méthode du coude"); plt.legend(); plt.show()
pd.DataFrame({"J (main)": J_par_k, "J (sklearn)": J_sklearn}, index=ks).round(1)


### III.2 Le coefficient de silhouette, à la main

Pour chaque point $i$ : $a(i)$ est la distance moyenne aux **autres** points de son cluster, $b(i)$ la distance moyenne aux points du cluster le plus proche **parmi les autres**, et
$$
s(i) = \frac{b(i)-a(i)}{\max(a(i),b(i))}, \qquad S = \frac{1}{n}\sum_i s(i).
$$
Calculez d'abord la matrice des distances entre tous les points (`scipy.spatial.distance.cdist`, ou à la main comme en I.1), puis $S$, et comparez à `sklearn.metrics.silhouette_score`.

In [ ]:
from scipy.spatial.distance import cdist
from sklearn.metrics import silhouette_score

def silhouette(X, etiquettes):
    D = cdist(X, X)
    groupes = np.unique(etiquettes)
    s = np.zeros(len(X))
    for i in range(len(X)):
        groupe_i = etiquettes[i]
        indices_i = np.flatnonzero(etiquettes == groupe_i)
        autres_i = indices_i[indices_i != i]
        a = D[i, autres_i].mean() if len(autres_i) else 0.0
        distances_moyennes = [D[i, etiquettes == groupe].mean() for groupe in groupes if groupe != groupe_i]
        b = min(distances_moyennes)
        s[i] = (b - a) / max(a, b) if max(a, b) else 0.0
    return s.mean()

etiq3 = meilleur_kmeans(Z_peng, 3)[0]
verifier("silhouette (k=3) = silhouette_score", silhouette(Z_peng, etiq3), silhouette_score(Z_peng, etiq3))


In [ ]:
S_par_k = {k: silhouette(Z_peng, meilleur_kmeans(Z_peng, k)[0]) for k in range(2, 9)}
plt.plot(list(S_par_k), list(S_par_k.values()), "o-")
plt.xlabel("k"); plt.ylabel("S(k)"); plt.title("Coefficient de silhouette"); plt.show()
pd.Series(S_par_k, name="S(k)").round(3)

✍️ **Question III.1.**
1. Où se situe le coude ? Est-il net ?
2. Quel $k$ maximise la silhouette ?
3. Les deux méthodes désignent-elles **3**, le nombre d'espèces ? Sans regarder les espèces, comment expliquez-vous la valeur trouvée ? (Repensez au nuage ACP de la Partie II.)

✍️ **Votre réponse :**

Le coude correspond à une forte baisse de l'inertie pour les premiers $k$, puis les gains deviennent plus faibles ; il n'est pas nécessairement net. La silhouette favorise le nombre de groupes dont la séparation géométrique est la meilleure, souvent $k=2$ sur ces données réduites. Ces critères décrivent la géométrie du nuage et ne sont pas obligés de retrouver le nombre biologique d'espèces.


---
# Partie IV — La classification ascendante hiérarchique  *(≈ 55 min)*

> **Question de la partie :** peut-on regrouper sans fixer $k$ à l'avance, et lire toute la hiérarchie des regroupements ?

### IV.1 En miniature : l'exemple du cours, à la main

7 points : $A(1,1)$, $B(1{,}5;2)$, $C(2;1{,}8)$, $D(6,6)$, $E(6{,}5;5{,}5)$, $F(7;6{,}5)$, $G(6{,}8;5)$. Le cours annonce, en **lien simple**, des fusions aux hauteurs $0{,}54$ ; $0{,}58$ ; $0{,}71$ ; $1{,}12$ ; $1{,}12$ ; $5{,}77$.

**Étape 1 : la matrice des distances.**

In [ ]:
noms = list("ABCDEFG")
P = np.array([(1, 1), (1.5, 2), (2, 1.8), (6, 6), (6.5, 5.5), (7, 6.5), (6.8, 5)])

D_cours = np.sqrt(((P[:, None, :] - P[None, :, :]) ** 2).sum(axis=2))
verifier("d(B, C) (cours : 0,54)", D_cours[1, 2], 0.5385)
pd.DataFrame(D_cours, index=noms, columns=noms).round(2)


**Étape 2 : l'algorithme ascendant.** On part de 7 clusters d'un seul point. À chaque étape, on fusionne les deux clusters les plus proches. La distance entre deux clusters $A$ et $B$ dépend du **critère de liaison** : `min` (lien simple), `max` (lien complet) ou `mean` (lien moyen) des distances entre leurs points.

Codez l'algorithme (version naïve, suffisante pour quelques points) : il renvoie la liste des fusions, avec leur hauteur.

In [ ]:
def cah(D, liaison=np.min):
    clusters = [[i] for i in range(len(D))]
    fusions = []
    while len(clusters) > 1:
        meilleure = None
        for a in range(len(clusters) - 1):
            for b in range(a + 1, len(clusters)):
                distance = liaison(D[np.ix_(clusters[a], clusters[b])])
                candidat = (distance, a, b)
                if meilleure is None or candidat < meilleure:
                    meilleure = candidat
        distance, a, b = meilleure
        gauche, droite = clusters[a], clusters[b]
        fusions.append((gauche.copy(), droite.copy(), float(distance)))
        clusters[a] = gauche + droite
        del clusters[b]
    return fusions

for gauche, droite, h in cah(D_cours, np.min):
    print(f"fusion de {''.join(noms[i] for i in gauche):>4s} et {''.join(noms[i] for i in droite):<4s} à la hauteur {h:.2f}")


**Étape 3 : comparaison avec `scipy`.** `scipy.cluster.hierarchy.linkage` renvoie une matrice dont la **3e colonne** contient les hauteurs de fusion.

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

for nom_liaison, f in [("single", np.min), ("complete", np.max), ("average", np.mean)]:
    h_main = np.array([h for _, _, h in cah(D_cours, f)])
    h_scipy = linkage(P, nom_liaison)[:, 2]
    verifier(f"hauteurs {nom_liaison}", h_main, h_scipy)

dendrogram(linkage(P, "single"), labels=noms)
plt.axhline(2.5, color="tab:red", ls="--"); plt.title("Lien simple (exemple du cours)"); plt.ylabel("hauteur de fusion"); plt.show()


### IV.2 ⚠️ Le critère de Ward : la formule du cours et celle de `scipy`

Le cours définit $d_{Ward}(A,B) = \dfrac{|A|\,|B|}{|A|+|B|}\,\|\mu_A - \mu_B\|^2$. Calculez-la pour la **première** fusion (deux points isolés, $B$ et $C$), et comparez à la première hauteur Ward de `scipy`.

In [ ]:
ward_cours_BC = (1 * 1 / (1 + 1)) * np.sum((P[1] - P[2]) ** 2)
h_ward_scipy = linkage(P, "ward")[:, 2]
print(f"d_Ward(B, C) selon le cours : {ward_cours_BC:.4f}")
print(f"première hauteur Ward scipy : {h_ward_scipy[0]:.4f}")


Les deux nombres diffèrent. Cherchez la relation (essayez $\sqrt{2\,d_{Ward}}$), puis vérifiez-la sur **toutes** les fusions avec votre fonction `cah`, en lui donnant comme liaison la formule de Ward calculée à partir des points.

In [ ]:
def cah_ward(P):
    clusters = [[i] for i in range(len(P))]
    hauteurs = []
    while len(clusters) > 1:
        meilleure = None
        for a in range(len(clusters) - 1):
            for b in range(a + 1, len(clusters)):
                A, B = clusters[a], clusters[b]
                moyenne_A = P[A].mean(axis=0)
                moyenne_B = P[B].mean(axis=0)
                distance = len(A) * len(B) / (len(A) + len(B)) * np.sum((moyenne_A - moyenne_B) ** 2)
                candidat = (distance, a, b)
                if meilleure is None or candidat < meilleure:
                    meilleure = candidat
        distance, a, b = meilleure
        hauteurs.append(distance)
        clusters[a] = clusters[a] + clusters[b]
        del clusters[b]
    return np.array(hauteurs)

h_ward_main = cah_ward(P)
verifier("scipy = sqrt(2 × d_Ward du cours)", h_ward_scipy, np.sqrt(2 * h_ward_main))


✍️ **Question IV.1.** Quelle est la relation entre les deux conventions ? Pourquoi $d_{Ward}$ est-il intéressant ? (Indice : calculez l'inertie intra-cluster $J$ de $\{B\}$, $\{C\}$, puis de $\{B, C\}$.) Est-ce gênant d'avoir deux conventions ?

✍️ **Votre réponse :**

La convention du cours mesure l'augmentation d'inertie : $d_W(A,B)=rac{|A||B|}{|A|+|B|}\|ar x_A-ar x_B\|^2$. SciPy affiche une hauteur égale à $\sqrt{2d_W}$, d'où les valeurs différentes mais exactement compatibles après conversion.


### IV.3 La CAH des manchots : quatre liaisons, quatre arbres

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
liens = {}
for ax, methode in zip(axes, ["single", "complete", "average", "ward"]):
    liens[methode] = linkage(P, methode)
    dendrogram(liens[methode], ax=ax, no_labels=True, color_threshold=None)
    ax.set_title(methode)
plt.show()


Pour chaque liaison, découpez l'arbre en **3 clusters** (`fcluster(..., t=3, criterion="maxclust")`) et regardez la **taille** des clusters obtenus.

In [ ]:
tailles = {}
for methode, arbre in liens.items():
    etiquettes = fcluster(arbre, t=3, criterion="maxclust")
    tailles[methode] = np.pad(np.bincount(etiquettes)[1:], (0, 3 - len(np.bincount(etiquettes)[1:])), constant_values=0)
pd.DataFrame(tailles, index=["cluster 1", "cluster 2", "cluster 3"])


✍️ **Question IV.2.** Que produit le lien **simple** ? Expliquez ce phénomène (appelé **effet de chaîne**) à partir de sa définition $d(A,B) = \min d(x,y)$. Quelle liaison donne les clusters les plus équilibrés ?

✍️ **Votre réponse :**

Le lien simple présente un effet de chaîne : des points peuvent être ajoutés successivement parce qu'une seule paire est proche, même si les groupes finaux sont peu compacts. Les liens complet et moyen favorisent des groupes plus équilibrés. Ward cherche à limiter l'augmentation d'inertie et produit généralement des groupes compacts, proches de l'objectif de K-means.


### IV.4 Où couper l'arbre de Ward ?

Le cours propose de couper au niveau de la **plus grande marche** : la plus grande différence entre deux hauteurs de fusion consécutives. Calculez les écarts entre les dernières hauteurs, et déduisez-en le $k$ suggéré.

In [ ]:
h = liens["ward"][:, 2]
dernieres = h[-8:]
ecarts = np.diff(dernieres)
for i, (hh, e) in enumerate(zip(dernieres[1:], ecarts)):
    k_avant = len(P) - i
    print(f"passage de {k_avant} à {k_avant - 1} clusters : hauteur {hh:6.2f}, marche = {e:6.2f}")

dendrogram(liens["ward"], no_labels=True, color_threshold=25)
plt.axhline(25, color="tab:red", ls="--"); plt.title("Ward : coupe à la hauteur 25"); plt.show()


✍️ **Votre réponse :**

On choisit une coupe juste avant une grande marche de hauteur : les fusions au-dessus de cette marche réunissent des groupes déjà relativement éloignés. Sur cet exemple, la coupe indiquée permet de conserver des groupes compacts ; le nombre exact de groupes doit être lu avec les hauteurs imprimées et le dendrogramme.


---
# Partie V — Vérité terrain : qu'ont retrouvé les algorithmes ?  *(≈ 15 min)*

> **Question de la partie :** on lève le voile sur les espèces. K-means et CAH les ont-ils retrouvées ?

Pour mesurer l'accord, on utilise la **pureté** : chaque cluster est associé à son espèce majoritaire, et l'on compte la proportion de manchots qui appartiennent à l'espèce majoritaire de leur cluster.

In [ ]:
def purete(clusters, vraies_classes):
    clusters = np.asarray(clusters)
    vraies_classes = np.asarray(vraies_classes)
    majoritaires = [pd.Series(vraies_classes[clusters == cluster]).value_counts().max()
                    for cluster in np.unique(clusters)]
    return sum(majoritaires) / len(vraies_classes)

liens_peng = {methode: linkage(Z_peng, methode)
              for methode in ["single", "ward"]}
partitions = {
    "K-means (k=3, réduit)": meilleur_kmeans(Z_peng, 3)[0],
    "K-means (k=3, NON réduit)": KMeans(n_clusters=3, n_init=10, random_state=0).fit(X_peng).labels_,
    "CAH single (k=3)": fcluster(liens_peng["single"], 3, "maxclust"),
    "CAH ward (k=3)": fcluster(liens_peng["ward"], 3, "maxclust"),
}
print(pd.Series({nom: purete(p, especes) for nom, p in partitions.items()}).round(3))
print("\nK-means (k=3, réduit) contre espèces:")
pd.crosstab(partitions["K-means (k=3, réduit)"], especes)


In [ ]:
print("CAH Ward (k=3) contre espèces:")
print(pd.crosstab(partitions["CAH ward (k=3)"], especes))
print("\nK-means contre CAH Ward:")
print(pd.crosstab(partitions["K-means (k=3, réduit)"], partitions["CAH ward (k=3)"]))


✍️ **Question V.1.**
1. Quelle espèce est retrouvée **parfaitement** par toutes les méthodes raisonnables ? Laquelle pose problème, et avec laquelle est-elle confondue ? Faites le lien avec le TP 3 (ACP contre ADL).
2. Pourquoi K-means sur données **non réduites** est-il si mauvais ?
3. K-means et Ward s'accordent-ils ? Lequel des deux se rapproche le plus des espèces ?

✍️ **Votre réponse :**

Sur les mesures réduites, Gentoo est généralement le groupe le mieux séparé. Adelie et Chinstrap se recouvrent davantage dans l'espace des mesures, donc leurs erreurs sont les plus fréquentes. Sans réduction, la masse corporelle, exprimée en grammes, domine les distances et masque les autres variables. Ward et K-means donnent des partitions géométriquement proches ; leur accord ne prouve toutefois pas que les clusters sont des espèces, seulement qu'ils optimisent des critères de compacité apparentés.


### 🔎 Bilan du TP

- K-means est un algorithme **simple et déterministe une fois initialisé**, qui converge toujours, mais vers un **optimum local** : environ un lancement sur quatre échoue ici, K-means++ « simple » n'y change rien, et seuls les redémarrages multiples le rendent fiable.
- Le « bon » nombre de clusters n'est pas une propriété des données qu'un indice révélerait : coude et silhouette décrivent la **géométrie**, et ici ils préfèrent 2 groupes là où la biologie en voit 3.
- La CAH donne **toute la hiérarchie** ; le critère de liaison change tout (effet de chaîne du lien simple) ; Ward est le cousin hiérarchique de K-means, avec une convention d'affichage à connaître.
- Sans étiquettes, un algorithme retrouve ce que la géométrie rend visible, ni plus, ni moins : c'est la limite fondamentale de l'apprentissage non supervisé.

> **Supervisé** (ADL, TP 3) : on sépare des groupes connus. **Non supervisé** (K-means, CAH) : on découvre des groupes inconnus, sans garantie qu'ils aient un sens métier.

---
# Partie VI — Livrable : clustering du jeu *Wine*

**À rendre individuellement sur Moodle, une semaine après la séance** : ce notebook complété.
Indiquez votre **nom, prénom et groupe** ci-dessous. Les réponses doivent refléter **votre propre compréhension** ; un texte générique ou généré automatiquement ne sera pas accepté.

**Nom, prénom, groupe :** …

On reprend les 178 vins du TP 3 (13 mesures), **sans** utiliser le cépage avant la tâche 4.

### Tâche 1 — K-means
1. Réduisez les 13 variables. Avec **vos** fonctions, tracez le coude ($k = 1 \dots 10$) et la silhouette ($k = 2 \dots 10$). Choisissez $k$ et justifiez.
2. Mesurez la fiabilité de l'initialisation aléatoire contre K-means++ (100 lancements chacun), comme en Partie II.

### Tâche 2 — CAH
1. Dendrogrammes avec `single`, `complete`, `average` et `ward`. Commentez l'effet de chaîne.
2. Ward : plus grande marche, $k$ suggéré, découpage.

### Tâche 3 — Le piège des unités
Refaites K-means **sans réduire** les variables. Quelle variable domine le calcul des distances ? Montrez-le (variances, centroïdes) et mesurez l'effet sur la pureté.

### Tâche 4 — Vérité terrain
Comparez vos partitions (K-means réduit, K-means non réduit, CAH Ward) au vrai cépage : tableaux croisés et puretés. Quel cépage est le plus difficile à isoler ? Projetez sur le plan ACP pour formuler une hypothèse.

### Tâche 5 — Retour aux manchots
Lancez K-means sur les manchots (réduits) avec $k = 6$, et croisez le résultat avec **l'espèce et le sexe** (`penguins[["species", "sex"]]`). Que découvrez-vous ? Qu'est-ce que cela dit du choix de $k$ ?

### Tâche 6 — Votre avis
Après ce TP, que répondriez-vous à quelqu'un qui affirme : « l'algorithme a trouvé 2 groupes, donc il y a 2 types de manchots » ?

In [ ]:
from sklearn.datasets import load_wine
wine = load_wine(as_frame=True)
X_wine = wine.data.values
cepages = np.array(wine.target_names)[wine.target]     # à n'utiliser qu'en tâche 4
Z_wine = (X_wine - X_wine.mean(axis=0)) / X_wine.std(axis=0, ddof=1)
print(X_wine.shape)

In [ ]:
# Tâche 1 — votre code ici

In [ ]:
# Tâche 2 — votre code ici

In [ ]:
# Tâche 3 — votre code ici

In [ ]:
# Tâche 4 — votre code ici

In [ ]:
# Tâche 5 — votre code ici

**Tâche 1 — choix de $k$ :** …

**Tâche 2 — commentaire :** …

**Tâche 3 — analyse :** …

**Tâche 4 — analyse :** …

**Tâche 5 — découverte :** …

**Tâche 6 — votre avis :** …